In [36]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import tiktoken
import numpy as np
from tqdm import tqdm
from google.genai import types
from google import genai
import time
from google.genai import errors
import os

In [2]:
df = pd.read_parquet("../../data/datasets/embeddings.parquet")

## OpenAI

In [3]:
enc = tiktoken.encoding_for_model("text-embedding-3-large")

In [4]:
import tiktoken
import pandas as pd

# -----------------------------
# Configuração
# -----------------------------
model = "text-embedding-3-large"
enc = tiktoken.encoding_for_model(model)

# As 3 colunas de texto originais
colunas_alvo = [
    "texto", 
    "texto_preprocessado", 
    "texto_preprocessado_sem_justificativa"
]

# -----------------------------
# Função de Contagem
# -----------------------------
def contar_tokens_coluna(df, col_name):
    if col_name not in df.columns:
        print(f"⚠️ Coluna '{col_name}' não encontrada no DataFrame.")
        return 0
        
    # Transforma em string, trata nulos e conta os tokens de cada linha
    total_tokens = df[col_name].fillna("").astype(str).apply(lambda x: len(enc.encode(x))).sum()
    return total_tokens

# -----------------------------
# Execução e Relatório
# -----------------------------
grand_total = 0
relatorio = []

print("📊 Contando tokens...\n")

for col in colunas_alvo:
    total_col = contar_tokens_coluna(df, col)
    grand_total += total_col
    relatorio.append({"Coluna": col, "Total de Tokens": f"{total_col:,}"})

# Exibe o resultado formatado em tabela
df_relatorio = pd.DataFrame(relatorio)
print(df_relatorio.to_string(index=False))

print("-" * 50)
print(f"🚀 TOTAL GERAL (3 Colunas): {grand_total:,} tokens")

📊 Contando tokens...

                               Coluna Total de Tokens
                                texto       3,325,969
                  texto_preprocessado       2,775,688
texto_preprocessado_sem_justificativa         738,879
--------------------------------------------------
🚀 TOTAL GERAL (3 Colunas): 6,840,536 tokens


In [5]:
import pandas as pd
import tiktoken

# -----------------------------
# Configuração
# -----------------------------
model = "text-embedding-3-large"
enc = tiktoken.encoding_for_model(model)
MAX_CONTEXT_TOKENS = 8192  # Limite da janela de contexto

colunas_alvo = [
    "texto", 
    "texto_preprocessado", 
    "texto_preprocessado_sem_justificativa"
]

# -----------------------------
# Função Padronizada de Higienização e Tokenização
# -----------------------------
def sanitize_and_count_tokens(text):
    """
    Usa exatamente a mesma regra do script de embedding:
    - Trata pd.isna, np.nan e None
    - Remove espaços externos (.strip())
    - Converte vazios em ' ' para evitar erro de string vazia
    """
    if pd.isna(text) or text is None:
        text = " "
    else:
        text = str(text).strip()
        if not text:
            text = " "
            
    return len(enc.encode(text))


# -----------------------------
# Função de Análise
# -----------------------------
def analisar_limite_coluna(df, col_name, max_tokens=MAX_CONTEXT_TOKENS):
    if col_name not in df.columns:
        print(f"⚠️ Coluna '{col_name}' não encontrada no DataFrame.")
        return None
        
    # Aplica a função padronizada linha a linha
    token_counts = df[col_name].apply(sanitize_and_count_tokens)
    
    total_linhas = len(df)
    excedentes = (token_counts > max_tokens).sum()
    pct_excedentes = (excedentes / total_linhas) * 100 if total_linhas > 0 else 0
    
    return {
        "Coluna": col_name,
        "Total Linhas": f"{total_linhas:,}",
        f"> {max_tokens} Tokens (Qtd)": f"{excedentes:,}",
        f"> {max_tokens} Tokens (%)": f"{pct_excedentes:.2f}%",
        "Max Tokens (Linha)": f"{token_counts.max():,}",
        "Total Tokens": f"{token_counts.sum():,}"
    }

# -----------------------------
# Execução e Relatório
# -----------------------------
relatorio_openai = []
print(f"📊 Analisando janela de contexto (Limite: {MAX_CONTEXT_TOKENS:,} tokens)...\n")

for col in colunas_alvo:
    resultado = analisar_limite_coluna(df, col)
    if resultado:
        relatorio_openai.append(resultado)

# Correção da variável no DataFrame
df_relatorio_openai = pd.DataFrame(relatorio_openai)
print(df_relatorio_openai.to_string(index=False))

📊 Analisando janela de contexto (Limite: 8,192 tokens)...

                               Coluna Total Linhas > 8192 Tokens (Qtd) > 8192 Tokens (%) Max Tokens (Linha) Total Tokens
                                texto        2,462                  12             0.49%             14,028    3,323,064
                  texto_preprocessado        2,462                   8             0.32%             12,467    2,775,688
texto_preprocessado_sem_justificativa        2,462                   2             0.08%              9,157      738,879


In [6]:
df_relatorio_openai.to_parquet("../../data/datasets/estatisticas_trunc_openai.parquet")

In [8]:
api_key = os.environ["OPENAI_API_KEY"]

In [10]:
# Inicializa cliente
client = OpenAI(api_key=api_key)

# Função para embedding
def get_embedding(texto, model="text-embedding-3-large"):
    response = client.embeddings.create(
        model=model,
        input=texto
    )
    return response.data[0].embedding

# Gerar embeddings com barra de progresso
emb = get_embedding("flamengo")

In [11]:
emb[:10]

[-0.029876708984375,
 0.0123138427734375,
 -0.00678253173828125,
 -0.018218994140625,
 -0.0189056396484375,
 0.047515869140625,
 -0.047027587890625,
 0.0247039794921875,
 -0.0226287841796875,
 0.0144805908203125]

In [12]:
len(emb)

3072

In [13]:
client = OpenAI(api_key=api_key)

MODEL = "text-embedding-3-large"
MAX_INPUT_TOKENS = 8192
MAX_REQUEST_TOKENS = 250_000  # Margem segura abaixo dos limites de TPM
MAX_BATCH_ITEMS = 2048       # Limite rígido da API da OpenAI por requisição

enc = tiktoken.encoding_for_model(MODEL)

def process_and_count_tokens(text, max_tokens=MAX_INPUT_TOKENS):
    """
    Higieniza strings vazias, tokeniza, trunca se necessário
    e retorna o texto processado junto à contagem de tokens.
    """
    text = str(text).strip() if text is not None else ""
    if not text:
        text = " "  # Evita erro de string vazia na API da OpenAI
    
    tokens = enc.encode(text)
    was_truncated = len(tokens) > max_tokens

    if was_truncated:
        tokens = tokens[:max_tokens]
        text = enc.decode(tokens)
        
    return text, len(tokens), was_truncated


def create_batches(texts, token_counts, max_batch_tokens=MAX_REQUEST_TOKENS, max_items=MAX_BATCH_ITEMS):
    batches = []
    current_texts = []
    current_indices = []
    current_tokens = 0

    for idx, (text, n_tokens) in enumerate(zip(texts, token_counts)):
        # Quebra o lote se atingir o limite de tokens OU o limite de itens da API
        if current_texts and (
            current_tokens + n_tokens > max_batch_tokens or len(current_texts) >= max_items
        ):
            batches.append((current_indices, current_texts))
            current_indices = []
            current_texts = []
            current_tokens = 0

        current_indices.append(idx)
        current_texts.append(text)
        current_tokens += n_tokens

    if current_texts:
        batches.append((current_indices, current_texts))

    return batches


def call_embedding_with_retry(batch_texts, max_retries=5):
    """Executa a requisição com backoff exponencial para evitar interrupções."""
    delay = 2
    for attempt in range(max_retries):
        try:
            return client.embeddings.create(
                model=MODEL,
                input=batch_texts
            )
        except (RateLimitError, APIError) as e:
            if attempt == max_retries - 1:
                raise e
            time.sleep(delay)
            delay *= 2


def embed_column_openai(df, input_col, output_col, normalize=False):
    df = df.copy()
    texts, token_counts, trunc_flags = [], [], []

    for text in df[input_col]:
        processed_text, n_tokens, truncated = process_and_count_tokens(text)
        texts.append(processed_text)
        token_counts.append(n_tokens)
        trunc_flags.append(truncated)

    print(f"\n[{input_col}]: {sum(trunc_flags)} textos truncados de {len(texts)} totais.")

    batches = create_batches(texts, token_counts)
    embeddings = [None] * len(texts)

    for indices, batch_texts in tqdm(batches, desc=f"Embedding {input_col}"):
        response = call_embedding_with_retry(batch_texts)

        # Garante o alinhamento ordenado pelo índice retornado pela API
        for item in response.data:
            idx = indices[item.index]
            emb = item.embedding
            if normalize:
                vec = np.asarray(emb, dtype=np.float32)
                norm = np.linalg.norm(vec)
                emb = (vec / norm).tolist() if norm > 0 else vec.tolist()
            embeddings[idx] = emb

    df[output_col] = embeddings
    return df

In [133]:
cols = [
    ("texto", "embedding__openai__text_embedding_3_large__texto"),
    ("texto_preprocessado", "embedding__openai__text_embedding_3_large__texto_preprocessado"),
    ("texto_preprocessado_sem_justificativa", "embedding__openai__text_embedding_3_large__texto_preprocessado_sem_justificativa"),
]

# ============================================================
# 4. Execução do Pipeline
# ============================================================
for col_in, col_out in cols:
    if col_in not in df.columns:
        print(f"⚠️ Coluna '{col_in}' não encontrada no DataFrame. Pulando...")
        continue

    print(f"\nIniciando processamento para: {col_in}")
    df = embed_column_openai(
        df=df,
        input_col=col_in,
        output_col=col_out,
        normalize=False,  # text-embedding-3 já retorna vetores de norma 1 por padrão
    )


Iniciando processamento para: texto

[texto]: 12 textos truncados de 2462 totais.


Embedding texto: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [01:56<00:00,  8.29s/it]



Iniciando processamento para: texto_preprocessado

[texto_preprocessado]: 8 textos truncados de 2462 totais.


Embedding texto_preprocessado: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [02:26<00:00, 12.23s/it]



Iniciando processamento para: texto_preprocessado_sem_justificativa

[texto_preprocessado_sem_justificativa]: 2 textos truncados de 2462 totais.


Embedding texto_preprocessado_sem_justificativa: 100%|██████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:43<00:00, 14.36s/it]


In [19]:
df.to_parquet("../../data/datasets/embeddings.parquet")

## Google

In [23]:
google_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

text = "What is the meaning of life?"

# 1. Sem prefixo
result1 = google_client.models.embed_content(
    model="gemini-embedding-2",
    contents=text
)
v1 = np.array(result1.embeddings[0].values)

# 2. Com prefixo de tarefa
result2 = google_client.models.embed_content(
    model="gemini-embedding-2",
    contents=f"task: sentence similarity | query: {text}"
)
v2 = np.array(result2.embeddings[0].values)

# Métricas de comparação
dim1, dim2 = len(v1), len(v2)
cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
dist_euclidiana = np.linalg.norm(v1 - v2)

print(f"Dimensões: {dim1} vs {dim2}")
print(f"Similaridade de Cosseno entre os dois: {cos_sim:.6f}")
print(f"Distância Euclidiana: {dist_euclidiana:.6f}")

Dimensões: 3072 vs 3072
Similaridade de Cosseno entre os dois: 0.762828
Distância Euclidiana: 0.688727


In [45]:
import pandas as pd
from tqdm import tqdm

# -----------------------------
# Configuração
# -----------------------------
MODEL_NAME = "gemini-embedding-2"  # ou "text-embedding-004"
MAX_CONTEXT_TOKENS = 8192

colunas_alvo = [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa",
]

# -----------------------------
# Função Auxiliar de Contagem
# -----------------------------
def contar_tokens_gemini(text, model=MODEL_NAME):
    if not text.strip():
        return 0
    try:
        response = google_client.models.count_tokens(
            model=model,
            contents=text
        )
        return response.total_tokens
    except Exception as e:
        print(f"⚠️ Erro ao contar tokens: {e}")
        return 0

# -----------------------------
# Função de Análise
# -----------------------------
def analisar_limite_coluna(df, col_name, max_tokens=MAX_CONTEXT_TOKENS):
    if col_name not in df.columns:
        print(f"⚠️ Coluna '{col_name}' não encontrada no DataFrame.")
        return None

    tqdm.pandas(desc=f"Contando {col_name}")
    
    # Contagem exata via API linha a linha
    token_counts = (
        df[col_name]
        .fillna("")
        .astype(str)
        .progress_apply(contar_tokens_gemini)
    )

    total_linhas = len(df)
    excedentes = int((token_counts > max_tokens).sum())
    pct_excedentes = (excedentes / total_linhas) * 100 if total_linhas > 0 else 0

    return {
        "Coluna": col_name,
        "Total Linhas": f"{total_linhas:,}",
        f"> {max_tokens} Tokens (Qtd)": f"{excedentes:,}",
        f"> {max_tokens} Tokens (%)": f"{pct_excedentes:.2f}%",
        "Max Tokens (Linha)": f"{int(token_counts.max()):,}",
        "Total Tokens": f"{int(token_counts.sum()):,}",
    }

# -----------------------------
# Execução e Relatório
# -----------------------------
# relatorio = []
# print(f"📊 Analisando janela de contexto (Limite: {MAX_CONTEXT_TOKENS:,} tokens)...\n")

# for col in colunas_alvo:
#     resultado = analisar_limite_coluna(df, col)
#     if resultado:
#         relatorio.append(resultado)

# df_relatorio = pd.DataFrame(relatorio)
# print("\n" + df_relatorio.to_string(index=False))

📊 Analisando janela de contexto (Limite: 8,192 tokens)...



Contando texto_preprocessado_sem_justificativa: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 2462/2462 [08:04<00:00,  5.08it/s]


                               Coluna Total Linhas > 8192 Tokens (Qtd) > 8192 Tokens (%) Max Tokens (Linha) Total Tokens
                                texto        2,462                  11             0.45%             12,791    3,202,285
                  texto_preprocessado        2,462                   8             0.32%             10,928    2,528,291
texto_preprocessado_sem_justificativa        2,462                   1             0.04%              8,799      730,461


In [46]:
df_relatorio.to_parquet("../../data/datasets/estatisticas_trunc_gemini.parquet")

In [47]:
texto_teste = "Bob Dylan "*4095
contar_tokens_gemini(texto_teste, "gemini-embedding-2")

8192

In [48]:
texto_teste = "Bob Dylan "*4095
print("Tokens em texto teste:", contar_tokens_gemini(texto_teste, "gemini-embedding-2"))

# conferência final
result = google_client.models.embed_content(
    model="gemini-embedding-2",
    contents=texto_teste
)

print(result.embeddings)

Tokens em texto teste: 8192
[ContentEmbedding(
  values=[
    0.0068271155,
    -0.006774195,
    -0.00959672,
    -0.017985377,
    0.02527402,
    <... 3067 more items ...>,
  ]
)]


In [53]:
texto_teste = "Bob Dylan "*10000
print("Tokens em texto teste:", contar_tokens_gemini(texto_teste, "gemini-embedding-2"))

# conferência final
result_2 = google_client.models.embed_content(
    model="gemini-embedding-2",
    contents=texto_teste
)

print(result_2.embeddings)

Tokens em texto teste: 20002
[ContentEmbedding(
  values=[
    0.0068271155,
    -0.006774195,
    -0.00959672,
    -0.017985377,
    0.02527402,
    <... 3067 more items ...>,
  ]
)]


In [54]:
result.embeddings[0].values == result_2.embeddings[0].values

True

In [55]:
MODEL = "gemini-embedding-2"  # ou "text-embedding-004"
MAX_BATCH_ITEMS = 20         # Limite recomendado de itens por requisição no Gemini

def sanitize_and_format_text(text, task_type=None):
    """
    Higieniza nulos/espaços e aplica o prefixo simétrico oficial:
    task: clustering | query: {content}
    task: sentence similarity | query: {content}
    """
    if pd.isna(text) or text is None:
        text = " "
    else:
        text = str(text).strip()
        if not text:
            text = " "

    if task_type:
        return f"task: {task_type} | query: {text}"
    return text


def create_batches(texts, batch_size=MAX_BATCH_ITEMS):
    batches = []
    total_len = len(texts)
    for i in range(0, total_len, batch_size):
        indices = list(range(i, min(i + batch_size, total_len)))
        batch_texts = texts[i : i + batch_size]
        batches.append((indices, batch_texts))
    return batches


def call_embedding_with_retry(batch_texts, max_retries=6):
    """
    Empacota cada texto em um types.Content para evitar agregação multimodal
    e aplica backoff exponencial contra erros transitórios e HTTP 429.
    """
    delay = 5
    # REGRA DA DOC: Cada texto precisa ser um Content individual
    contents = [
        types.Content(parts=[types.Part.from_text(text=t)])
        for t in batch_texts
    ]

    for attempt in range(max_retries):
        try:
            response = google_client.models.embed_content(
                model=MODEL,
                contents=contents,
            )
            time.sleep(0.4)  # Respiro contra limites de requisições por minuto
            return response.embeddings
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"\n⚠️ Tentativa {attempt + 1}/{max_retries} falhou ({e}). Aguardando {delay}s...")
            time.sleep(delay)
            delay *= 2


def embed_column(df, input_col, output_col, task_type=None, normalize=False):
    df = df.copy()

    texts = [
        sanitize_and_format_text(text, task_type=task_type)
        for text in df[input_col]
    ]

    batches = create_batches(texts, batch_size=MAX_BATCH_ITEMS)
    embeddings = [None] * len(texts)

    desc_label = f"Gemini ({task_type or 'raw'}) -> {input_col}"
    for indices, batch_texts in tqdm(batches, desc=desc_label):
        emb_results = call_embedding_with_retry(batch_texts)

        if len(emb_results) != len(batch_texts):
            raise ValueError(
                f"Erro de agregação: enviados {len(batch_texts)} textos, recebidos {len(emb_results)} vetores."
            )

        for idx, emb_obj in zip(indices, emb_results):
            emb = emb_obj.values
            if normalize:
                vec = np.asarray(emb, dtype=np.float32)
                norm = np.linalg.norm(vec)
                emb = (vec / norm).tolist() if norm > 0 else vec.tolist()
            embeddings[idx] = emb

    df[output_col] = embeddings
    return df

In [60]:
# ============================================================
# Definição das Colunas e Tarefas a Processar
# Formato: (coluna_entrada, coluna_saida, task_type)
# Padrão: embedding__provedor__modelo__tarefa__tipo_de_texto
# ============================================================
cols = [
    # 1. Modo Raw (Sem prompt)
    (
        "texto",
        "embedding__gemini__gemini_embedding_2__raw__texto",
        None
    ),
    (
        "texto_preprocessado",
        "embedding__gemini__gemini_embedding_2__raw__texto_preprocessado",
        None
    ),
    (
        "texto_preprocessado_sem_justificativa",
        "embedding__gemini__gemini_embedding_2__raw__texto_preprocessado_sem_justificativa",
        None
    ),

    # 2. Modo Clustering
    (
        "texto",
        "embedding__gemini__gemini_embedding_2__clustering__texto",
        "clustering"
    ),
    (
        "texto_preprocessado",
        "embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado",
        "clustering"
    ),
    (
        "texto_preprocessado_sem_justificativa",
        "embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado_sem_justificativa",
        "clustering"
    ),

    # 3. Modo Semantic Similarity (STS)
    (
        "texto",
        "embedding__gemini__gemini_embedding_2__sts__texto",
        "sentence similarity"
    ),
    (
        "texto_preprocessado",
        "embedding__gemini__gemini_embedding_2__sts__texto_preprocessado",
        "sentence similarity"
    ),
    (
        "texto_preprocessado_sem_justificativa",
        "embedding__gemini__gemini_embedding_2__sts__texto_preprocessado_sem_justificativa",
        "sentence similarity"
    ),
]

# ============================================================
# Execução do Pipeline
# ============================================================
for col_in, col_out, task in cols:
    if col_in not in df.columns:
        print(f"⚠️ Coluna de entrada '{col_in}' não encontrada no DataFrame. Pulando...")
        continue

    if col_out in df.columns:
        print(f"⏩ Coluna de saída '{col_out}' já existe. Pulando...")
        continue

    print(f"\nIniciando processamento: '{col_in}' -> '{col_out}' [Task: {task or 'raw'}]")
    df = embed_column_gemini(
        df=df,
        input_col=col_in,
        output_col=col_out,
        task_type=task,
        normalize=False,
    )
    print(f"✅ Concluído: {col_out}")

print("\n🎉 Todas as colunas foram processadas com sucesso!")

⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__raw__texto' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado_sem_justificativa' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__clustering__texto' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__clustering__texto_preprocessado_sem_justificativa' já existe. Pulando...
⏩ Coluna de saída 'embedding__gemini__gemini_embedding_2__sts__texto' já existe. Pulando...

Iniciando processamento: 'texto_preprocessado' -> 'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado' [Task: sentence similarity]


Gemini (sentence similarity) -> texto_preprocessado: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [04:01<00:00,  1.95s/it]


✅ Concluído: embedding__gemini__gemini_embedding_2__sts__texto_preprocessado

Iniciando processamento: 'texto_preprocessado_sem_justificativa' -> 'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado_sem_justificativa' [Task: sentence similarity]


Gemini (sentence similarity) -> texto_preprocessado_sem_justificativa: 100%|██████████████████████████████████████████████████████████████████████████████| 124/124 [02:54<00:00,  1.41s/it]

✅ Concluído: embedding__gemini__gemini_embedding_2__sts__texto_preprocessado_sem_justificativa

🎉 Todas as colunas foram processadas com sucesso!


In [68]:
df.to_parquet("../../data/datasets/embeddings.parquet")